## Preface

This notebook will be used for developing Automated Synapse Tuning.

Greg Glickert has already done good work to develop location-independent STP and PSC tuning using scipy.optimize

This notebook will extend that application to include location-dependent PSC tuning.

This notebook will attempt to use scipy.optimize.differential_evolution.

## Dependencies

In [ ]:
# sudo apt update
# sudo apt install python3-pip
# pip3 install numpy
# pip3 install 
# pip3 install scipy

## Example of differential_evolution

In [3]:
import time
import numpy as np
from scipy.optimize import minimize, differential_evolution
import os
import multiprocessing
from multiprocessing import Manager

mgr = Manager()
pid_list = mgr.list()

# Multimodal test function: Rastrigin
def rastrigin(x):
    pid = os.getpid()
    # print(f"Evaluating in PID {pid}")
    if pid not in pid_list:
        pid_list.append(pid)
    A = 10
    return A * len(x) + sum(xi**2 - A * np.cos(2 * np.pi * xi) for xi in x)

def show_pool(x, convergence):
    kids = multiprocessing.active_children()
    print("Active children:", [p.name for p in kids])
    return False  # don’t halt

# Problem setup
dim    = 4
bounds = [(-5.12, 5.12)] * dim
x0     = np.zeros(dim)  # starting guess at the known global minimum

# 1) Local solver (BFGS) from x0
t0 = time.perf_counter()
res_local = minimize(rastrigin, x0, method='BFGS')
t_local = time.perf_counter() - t0

# 2) Global solver (Differential Evolution)
t1 = time.perf_counter()
res_global = differential_evolution(
    rastrigin,
    bounds,
    workers=4,            # parallel evaluation of the population
    updating='deferred',   # allows batching of function calls
    disp=True,
    # callback=show_pool # show the pool using a callback function
)
t_global = time.perf_counter() - t1

# Report
print("Worker PIDs:", list(pid_list))
print("Count of distinct workers:", len(pid_list))

print(f"Local solver:")
print(f"  x = {res_local.x}")
print(f"  fun = {res_local.fun:.6e}")
print(f"  time = {t_local:.4f} s\n")

print(f"Global solver:")
print(f"  x = {res_global.x}")
print(f"  fun = {res_global.fun:.6e}")
print(f"  time = {t_global:.4f} s")


differential_evolution step 1: f(x)= 26.353859676013236
differential_evolution step 2: f(x)= 26.353859676013236
differential_evolution step 3: f(x)= 22.189470187034942
differential_evolution step 4: f(x)= 20.272138307179596
differential_evolution step 5: f(x)= 20.272138307179596
differential_evolution step 6: f(x)= 15.038276881425123
differential_evolution step 7: f(x)= 12.008352392253414
differential_evolution step 8: f(x)= 12.008352392253414
differential_evolution step 9: f(x)= 12.008352392253414
differential_evolution step 10: f(x)= 12.008352392253414
differential_evolution step 11: f(x)= 12.008352392253414
differential_evolution step 12: f(x)= 11.75733626347023
differential_evolution step 13: f(x)= 11.263605690351277
differential_evolution step 14: f(x)= 11.263605690351277
differential_evolution step 15: f(x)= 11.263605690351277
differential_evolution step 16: f(x)= 9.005169587106394
differential_evolution step 17: f(x)= 9.005169587106394
differential_evolution step 18: f(x)= 9.005

## Tuning location-dependent PSCs.